# Step 2 - Prototype 2: With Train Types and Big M Speed Control

This prototype extends Step 2 Prototype 1 with:
- **Train Type System**: Express, Regional, Freight with different speeds using the Big M Notation
- **Station Proximity Speed Reduction**: Trains  should now also slow down near stations

We maintain bidirectional train movement and add intelligent speed management based on train type and location.

**Remarks**

If we implement rail switches or segments after segments for the boolean that checks if a block is next to a station, we get problems. The Logic right now builds on that after and before every segment we have a station.

In [ ]:
# Import Gurobi
import gurobipy as gp
from gurobipy import GRB

import numpy as np

In [ ]:
# Initialize the Model
model = gp.Model("Train_Network_BigM")

## Track System Definition

In [ ]:
# Define the track system using dictionaries to have homogeneous lists and allow for precise track building

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 1},
    {"type": "station", "capacity": 2},
    {"type": "segment", "length": 2, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [ ]:
# Parsing the track blueprint and generating necessary data
block_capacities = [] #List with all capacities
is_station = [] #boolean list for all stations and segments, true iff station

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

## Train Types and Speed Profiles


In [ ]:
# Define train types with their characteristics
train_types = {
    'express': {
        'max_speed': 3,              # blocks per time unit
        'station_speed': 1,          # reduced speed near stations
        'color': 'red'
    },
    'regional': {
        'max_speed': 2,              # blocks per time unit
        'station_speed': 1,          # reduced speed near stations
        'color': 'blue'
    },
    'freight': {
        'max_speed': 1,              # blocks per time unit
        'station_speed': 1,          # reduced speed near stations (already slow)
        'color': 'green'
    }
}

train_type_names = list(train_types.keys()) #To access the different train types
print(f"Train types available: {train_type_names}")

## Train Schedules and Information

In [ ]:
# Train information with passenger counts and train types
train_information = [
    # Train 0 (RE-like)
    {
        "info": {"passenger_count": 100, "train_type": "regional"},
        "schedule": [
            # First journey
            {"station": 0, "departure": 3},
            {"station": 1, "arrival": 10, "departure": 13},
            # Second journey
            {"station": 2, "arrival": 12, "departure": 20},
            {"station": 1, "arrival": 27, "departure": 29},
            {"station": 0, "arrival": 34}
        ]
    },
    # Train 1 (ICE-like)
    {
        "info": {"passenger_count": 250, "train_type": "express"},
        "schedule": [
            # First journey
            {"station": 0, "departure": 5},
            # Second journey
            {"station": 2, "arrival": 13, "departure": 18, "dwell": 5},
            {"station": 0, "arrival": 29}
        ]
    },
]

In [ ]:
# Number of time steps over which we optimize
time_horizon = 50

# Deriving some additional variables
num_trains = len(train_information)
num_blocks = len(block_capacities) #number of blocks
num_stations = sum(is_station) #1 for stations else 0

# List of all block indices that are stations
station_block_indices = [block for block, station_idx in enumerate(is_station) if station_idx]

print(f"Number of trains: {num_trains}")
print(f"Number of blocks: {num_blocks}")
print(f"Number of stations: {num_stations}")
print(f"Station block indices: {station_block_indices}")




## Mark Blocks next to a train station



In [ ]:
# Extract train types from train information
train_types_assignment = []
for i in range(num_trains):
    train_types_assignment.append(train_information[i]["info"]["train_type"])

# Define blocks near stations (segment-aware: only mark stations and segment boundaries)
is_near_station = np.zeros(num_blocks, dtype=bool)
current_block = 0

for item in track_blueprint:
    if item["type"] == "station":
        is_near_station[current_block] = True
        current_block += 1
    elif item["type"] == "segment":
        length = item["length"]
        # Mark first and last block of segment
        is_near_station[current_block] = True  # first block should be next to station
        is_near_station[current_block + length - 1] = True  # last block should be next to station
        current_block += length

print(f"Train type assignments: {train_types_assignment}")
print(f"Blocks near stations: {np.where(is_near_station)[0]}")

## Decision Variables

In [ ]:
# Position variables for bidirectional movement
x_fwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_fwd")
x_bwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_bwd")

# Big M binary variables for train type assignment
# y[i, train_type] = 1 if train i is of given train_type
y = {}
for i in range(num_trains):
    for train_type in train_type_names:
        y[i, train_type] = model.addVar(vtype=GRB.BINARY, 
                                        name=f"y_{i}_{train_type}")

# Speed variables for each train at each time step (forward and backward directions)
speed_fwd = {}
speed_bwd = {}
for i in range(num_trains):
    for t in range(time_horizon - 1):
        speed_fwd[i, t] = model.addVar(vtype=GRB.INTEGER, lb=0, 
                                        name=f"speed_fwd_{i}_{t}")
        speed_bwd[i, t] = model.addVar(vtype=GRB.INTEGER, lb=0, 
                                        name=f"speed_bwd_{i}_{t}")

# Binary variable to track if train is near a station at position j
near_station_bin = {}
for i in range(num_trains):
    for t in range(time_horizon):
        for j in range(num_blocks):
            if is_near_station[j]:
                near_station_bin[i, t, j] = model.addVar(vtype=GRB.BINARY, 
                                                          name=f"near_st_{i}_{t}_{j}")

model.update()
print(f"Added {len(y)} train type assignment variables")
print(f"Added {len(speed_fwd)} forward speed variables and {len(speed_bwd)} backward speed variables")
print(f"Added {len(near_station_bin)} station proximity indicators")

## Big M Constraints for Train Type Enforcement

In [ ]:
# Big M constant ( just our upper bound on speed (fastest train))
M = max([train_types[tt]['max_speed'] for tt in train_type_names])

# Each train must be assigned exactly one type
for i in range(num_trains):
    model.addConstr(
        gp.quicksum(y[i, tt] for tt in train_type_names) == 1,
        name=f"assign_type_{i}"
    )

# Pre-assign train types based on train_types_assignment
for i in range(num_trains):
    assigned_type = train_types_assignment[i]
    model.addConstr(
        y[i, assigned_type] == 1,
        name=f"force_type_{i}_{assigned_type}"
    )

# Speed constraint (forward): use Big M to enforce max speed based on train type
for i in range(num_trains):
    for t in range(time_horizon - 1):
        for tt in train_type_names:
            max_speed_for_type = train_types[tt]['max_speed']
            # If train i is of type tt (y[i, tt] = 1), then speed_fwd[i, t] <= max_speed_for_type
            # Otherwise: speed_fwd[i, t] <= M (relaxed)
            model.addConstr(
                speed_fwd[i, t] <= max_speed_for_type + M * (1 - y[i, tt]),
                name=f"max_speed_fwd_type_{i}_{t}_{tt}"
            )
            
            # Same for backward direction
            model.addConstr(
                speed_bwd[i, t] <= max_speed_for_type + M * (1 - y[i, tt]),
                name=f"max_speed_bwd_type_{i}_{t}_{tt}"
            )

print(f"Added train type assignment constraints")
print(f"Big M value: {M}")

## Station Proximity Speed Reduction

In [ ]:
# Link position variables and station proximity indicators
for i in range(num_trains):
    for t in range(time_horizon):
        for j in range(num_blocks):
            if is_near_station[j]:
                # near_station_bin[i, t, j] = 1 if train i is at/near block j at time t
                model.addConstr(
                    near_station_bin[i, t, j] <= x_fwd[i, j, t] + x_bwd[i, j, t],
                    name=f"detect_near_station_fwd_bwd_{i}_{t}_{j}"
                )

# Speed reduction near stations using Big M (forward direction)
for i in range(num_trains):
    for t in range(time_horizon - 1):
        for j in range(num_blocks):
            if is_near_station[j]:
                for tt in train_type_names:
                    station_speed_for_type = train_types[tt]['station_speed']
                    # If train i is at a station-proximity block and is of type tt:
                    # speed_fwd[i, t] <= station_speed_for_type
                    model.addConstr(
                        speed_fwd[i, t] <= station_speed_for_type + M * (1 - y[i, tt]) + M * (1 - near_station_bin[i, t, j]),
                        name=f"station_speed_reduction_fwd_{i}_{t}_{j}_{tt}"
                    )

# Speed reduction near stations using Big M (backward direction)
for i in range(num_trains):
    for t in range(time_horizon - 1):
        for j in range(num_blocks):
            if is_near_station[j]:
                for tt in train_type_names:
                    station_speed_for_type = train_types[tt]['station_speed']
                    # If train i is at a station-proximity block and is of type tt:
                    # speed_bwd[i, t] <= station_speed_for_type
                    model.addConstr(
                        speed_bwd[i, t] <= station_speed_for_type + M * (1 - y[i, tt]) + M * (1 - near_station_bin[i, t, j]),
                        name=f"station_speed_reduction_bwd_{i}_{t}_{j}_{tt}"
                    )

print(f"Added station proximity speed reduction constraints")

## Movement and Turnaround Constraints

In [ ]:
# Movement Constraints with Bidirectional Movement and Turnaround Support

for i in range(num_trains):
    # Extract the spawn station id and spawn time (first departure time)
    spawn_station_id = train_information[i]["schedule"][0]["station"]
    spawn_time = train_information[i]["schedule"][0]["departure"]

    # At every time point, we need to differentiate between different scenarios
    for k in range(time_horizon):

        # If the train hasn't left yet, it's not yet on the grid
        if k < spawn_time:
            pass
        
        # Here the train spawns, and we fix a position (the block that corresponds to the first station)
        elif k == spawn_time:
            block_index = station_block_indices[spawn_station_id]
            next_station_id = train_information[i]["schedule"][1]["station"]

            # Determine the initial direction of the train
            if spawn_station_id < next_station_id:
                # Moving right -> Spawn in Forward tensor
                model.addConstr(x_fwd[i, block_index, k] == 1, name=f"SpawnFwd_Train{i}_Time{k}_Block{block_index}")
            else:
                # Moving left -> Spawn in Backward tensor
                model.addConstr(x_bwd[i, block_index, k] == 1, name=f"SpawnBwd_Train{i}_Time{k}_Block{block_index}")

        else:
            # Handle back-looking approach for both channels (tensors)
            for j in range(num_blocks):

                # Forward channel (move 'right')
                # Calculate if this block is a station to allow turning around (switching from Bwd to Fwd)
                turnaround_fwd = x_bwd[i, j, k - 1] if is_station[j] else 0

                if j == 0:
                    model.addConstr(x_fwd[i, 0, k] <= x_fwd[i, 0, k - 1] + turnaround_fwd, 
                                    name=f"MoveFwd_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_fwd[i, j, k] <= x_fwd[i, j, k - 1] + x_fwd[i, j - 1, k - 1] + turnaround_fwd, 
                                    name=f"MoveFwd_Train{i}_Time{k}_Block{j}")

                # Backward channel (move 'left')
                # Allow switching from Fwd to Bwd
                turnaround_bwd = x_fwd[i, j, k - 1] if is_station[j] else 0

                if j == num_blocks - 1:
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + turnaround_bwd, 
                                    name=f"MoveBwd_Train{i}_Time{k}_Block{num_blocks-1}")
                else:
                    # General back-look constraint (train must have been here or in the next block)
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + x_bwd[i, j + 1, k - 1] + turnaround_bwd, 
                                    name=f"MoveBwd_Train{i}_Time{k}_Block{j}")

print(f"Added movement constraints with turnaround support")

## Intermediate Station and Departure Time Constraints

In [ ]:
# Intermediate Station Constraints: Enforce arrival and departure times

for i in range(num_trains):
    schedule = train_information[i]["schedule"]

    # Loop through all stops except the spawn station
    for stop_idx in range(1, len(schedule)):
        current_stop = schedule[stop_idx]

        # If there is no departure time, it's the final destination
        if "departure" not in current_stop:
            continue

        # Check for malformed schedules
        if stop_idx + 1 >= len(schedule):
            print(f"Warning: Train {i} has a departure time at its final stop. Double check schedules...")
            continue

        # Extract variables
        departure_time = current_stop["departure"]
        station_id = current_stop["station"]
        station_block = station_block_indices[station_id]
        next_station_id = schedule[stop_idx + 1]["station"]

        # Prohibit departure before departure_time by blocking the next block
        if next_station_id > station_id:
            # Moving forward
            next_block = station_block + 1
            if next_block < num_blocks:
                for k in range(departure_time):
                    model.addConstr(x_fwd[i, next_block, k] == 0, 
                                    name=f"DepWallFwd_Train{i}_Block{next_block}_Time{k}")

        else:
            # Moving backward
            next_block = station_block - 1
            if next_block >= 0:
                for k in range(departure_time):
                    model.addConstr(x_bwd[i, next_block, k] == 0, 
                                    name=f"DepWallBwd_Train{i}_Block{next_block}_Time{k}")

print(f"Added intermediate station and departure time constraints")

## Uniqueness Constraint


In [ ]:
# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (departure times), as before a train does not exist on any block
for i in range(num_trains):

    # Wir greifen jetzt korrekt auf das neue Listen-Format im Schedule zu
    spawn_time = train_information[i]["schedule"][0]["departure"]

    for k in range(time_horizon):

        # Summe über alle Blöcke j für Vorwärts- UND Rückwärtsrichtung
        active_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track
        if k < spawn_time:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, the train must be on exactly one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

print("Added uniqueness and spawn constraints")

## Capacity Constraints

In [ ]:
# Capacity constraints: limit trains per block
for k in range(time_horizon):
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for i in range(num_trains))
        model.addConstr(occupied_tracks <= block_capacities[j], name=f"Capacity_Block{j}_Time{k}")

print(f"Added capacity constraints")

## Objective: Minimize Weighted Lateness at Destination Stations

In [ ]:
# Objective: Minimize weighted lateness at destination stations

# Extract destination information for each train
destination_info = []
for i in range(num_trains):
    schedule = train_information[i]["schedule"]
    dest_station_id = schedule[-1]["station"]
    dest_block = station_block_indices[dest_station_id]
    target_arrival = schedule[-1]["arrival"]
    passenger_count = train_information[i]["info"]["passenger_count"]
    destination_info.append({
        "block": dest_block,
        "arrival_time": target_arrival,
        "weight": passenger_count
    })

# Build objective terms: sum of (delay * weight) for each train
objective_terms = []

for i in range(num_trains):
    dest_info = destination_info[i]
    dest_block = dest_info["block"]
    target_time = dest_info["arrival_time"]
    weight = dest_info["weight"]

    # Add penalty for every time step after the target arrival
    for k in range(target_time + 1, time_horizon):
        delay_penalty = k - target_time
        # Transition: train arrives at destination when it moves to dest_block at time k
        transition = x_fwd[i, dest_block, k] - x_fwd[i, dest_block, k - 1]
        transition += x_bwd[i, dest_block, k] - x_bwd[i, dest_block, k - 1]
        
        objective_terms.append(weight * delay_penalty * transition)

model.setObjective(gp.quicksum(objective_terms), GRB.MINIMIZE)
print(f"Added objective: minimize weighted lateness at destination")

## Model Summary

In [ ]:
print("\n=== Model Summary ===")
print(f"Number of variables: {model.numVars}")
print(f"Number of constraints: {model.numConstrs}")
print(f"\nTrain Type Assignments:")
for i, tt in enumerate(train_types_assignment):
    passenger_count = train_information[i]["info"]["passenger_count"]
    max_speed = train_types[tt]['max_speed']
    print(f"  Train {i}: {tt:10s} (passengers: {passenger_count:3d}, max speed: {max_speed} blocks/time)")
print(f"\nStation blocks: {station_block_indices}")
print(f"Blocks near stations: {np.where(is_near_station)[0]}")